In [ ]:
# CYR-GPU-010 — CELL 0: FREEZE / TEST / CALIBRATE / RESOLVE
import json, os, subprocess, sys
from pathlib import Path
REPO = Path('/content/An-Ra-the-new-AGI')
BRANCH = 'cymek-500m-readiness'
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
PREREG_PATH = REPO / 'docs/cymek/experiments/CYR-GPU-010/PREREGISTRATION.json'
if not PREREG_PATH.exists():
    raise RuntimeError('CYR-GPU-010 is not preregistered yet')
PREREG = json.loads(PREREG_PATH.read_text('utf-8'))
EXTERNAL = Path('/content/CYR-GPU-010-PREREGISTRATION.json')
EXTERNAL.write_text(json.dumps(PREREG, indent=2, sort_keys=True), encoding='utf-8')
EXECUTABLE_SHA = PREREG['executable_sha']
subprocess.run(['git','checkout','-q',EXECUTABLE_SHA], check=True)
HEAD = subprocess.run(['git','rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert HEAD == EXECUTABLE_SHA, (HEAD, EXECUTABLE_SHA)
for relative, expected_blob in PREREG['executable_blobs'].items():
    actual_blob = subprocess.run(['git','hash-object',relative], check=True, capture_output=True, text=True).stdout.strip()
    assert actual_blob == expected_blob, f'blob mismatch: {relative}'
print('frozen executable verified', HEAD)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers','pytest'], check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu010.py','-q'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CYR-GPU-010 requires a Google Colab CUDA GPU')
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from anra_v5.cyr_gpu010_run import production_tokenizer, calibrate_candidates
from v5_experiments import cyr_gpu010 as core
tokenizer, identity = production_tokenizer(REPO)
assert identity['artifact_sha256'] == PREREG['tokenizer']['artifact_sha256']
splits = core.render_t2_worlds()
manifest = core.build_data_manifest(splits)
core.assert_manifest_sha(manifest)
assert manifest['sha256'] == PREREG['data']['data_manifest_sha256_full']
audit = core.commutation_audit(splits, tv_bound=0.20)
assert audit['commutation_free'], audit['findings']
registry = core.proxy_registry(vocab_size=identity['vocabulary_size'])
special = {'pad_id':identity['pad_id'],'bos_id':identity['bos_id'],'eos_id':identity['eos_id']}
CALIBRATIONS = calibrate_candidates(registry=registry, tokenizer=tokenizer, torch=torch, device=DEVICE, special=special, train_rows=splits['train'], eval_rows=splits['dev_controller'])
Path('/content/CYR-GPU-010-CALIBRATIONS.json').write_text(json.dumps(CALIBRATIONS, indent=2, sort_keys=True))
print('CALIBRATION SUMMARY')
for name,r in CALIBRATIONS.items():
    print(name, r.get('status'), 'params=',r.get('parameters'),'updates/s=',round(float(r.get('training_updates_per_sec',0)),3),'train_tok/s=',round(float(r.get('training_real_tokens_per_sec',0)),1),'gen_ex/s=',round(float(r.get('generation_examples_per_sec',0)),1))
RESOLVED = core.resolve_from_calibrations(CALIBRATIONS)
core.validate_resolved(RESOLVED)
Path('/content/CYR-GPU-010-RESOLVED.json').write_text(json.dumps(RESOLVED, indent=2, sort_keys=True))
print(json.dumps(RESOLVED, indent=2))
print('CYR-GPU-010 PREEXECUTION GATE: PASS')


In [ ]:
# CYR-GPU-010 — CELL 1: LONG ONE-SHOT RUN / DRIVE DURABILITY
import json, sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/An-Ra-the-new-AGI')
sys.path.insert(0, str(REPO))
DEVICE = torch.device('cuda')
assert torch.cuda.is_available() and DEVICE.type == 'cuda'
PREREG = json.loads(Path('/content/CYR-GPU-010-PREREGISTRATION.json').read_text())
CALIBRATIONS = json.loads(Path('/content/CYR-GPU-010-CALIBRATIONS.json').read_text())
RESOLVED = json.loads(Path('/content/CYR-GPU-010-RESOLVED.json').read_text())
OUT = Path('/content/drive/MyDrive/CYMEK/CYR-GPU-010')
OUT.mkdir(parents=True, exist_ok=True)
from anra_v5.cyr_gpu010_run import run_campaign
try:
    campaign = run_campaign(repo=REPO, out=OUT, preregistration=PREREG, resolved=RESOLVED, calibrations=CALIBRATIONS, torch=torch, device=DEVICE, progress=lambda msg: print(msg, flush=True))
    print('DONE:', campaign['status'])
    print('VERDICT:', campaign.get('decision',{}).get('verdict'))
    print('G90 CONFIRM UPDATE:', campaign.get('acquisition',{}).get('g90_confirm_update'))
    print('BUNDLE:', campaign['bundle']['path'])
except Exception as exc:
    print('RUN FAILED, but partial evidence was packaged on Drive:', repr(exc))
    print('OUT:', OUT)
    raise


In [ ]:
# CYR-GPU-010 — CELL 2: VERIFY / DOWNLOAD
import hashlib, json
from pathlib import Path
from google.colab import files
OUT = Path('/content/drive/MyDrive/CYMEK/CYR-GPU-010')
receipt = json.loads((OUT/'campaign_receipt.json').read_text('utf-8'))
bundle = Path(receipt['bundle']['path'])
assert bundle.exists(), bundle
actual = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert actual == receipt['bundle']['sha256'], 'bundle hash mismatch'
print('verified', bundle.name, actual)
print('status:', receipt['status'], 'verdict:', receipt.get('decision',{}).get('verdict'))
print('proxy:', receipt.get('resolved',{}).get('proxy'), 'updates:', receipt.get('acquisition',{}).get('updates'), 'G90:', receipt.get('acquisition',{}).get('g90_confirm_update'))
files.download(str(bundle))
